In [1]:
import pandas as pd

In [2]:
matches = pd.read_csv("data/matches_original.csv")
print(matches.head())

           MatchId                                            Player1  \
0  EUW1_7266803975  ['champion_10', 'TOP', 'tier_BRONZE', 'rank_II...   
1  EUW1_7266792870  ['champion_54', 'TOP', 'tier_BRONZE', 'rank_II...   
2  EUW1_7265401358  ['champion_10', 'TOP', 'tier_BRONZE', 'rank_II...   
3  EUW1_7263778355  ['champion_48', 'TOP', 'UNRANKED', 'UNRANKED',...   
4  EUW1_7263730042  ['champion_82', 'TOP', 'tier_BRONZE', 'rank_II...   

                                             Player2  \
0  ['champion_72', 'JUNGLE', 'tier_IRON', 'rank_I...   
1  ['champion_154', 'JUNGLE', 'tier_IRON', 'rank_...   
2  ['champion_72', 'JUNGLE', 'tier_IRON', 'rank_I...   
3  ['champion_57', 'JUNGLE', 'UNRANKED', 'UNRANKE...   
4  ['champion_72', 'JUNGLE', 'tier_BRONZE', 'rank...   

                                             Player3  \
0  ['champion_901', 'MIDDLE', 'tier_SILVER', 'ran...   
1  ['champion_61', 'MIDDLE', 'tier_IRON', 'rank_I...   
2  ['champion_105', 'MIDDLE', 'UNRANKED', 'UNRANK...   


In [3]:
print(len(matches))
print(matches.columns)
print(matches.MatchId.str.split("_").str[0].unique())

10000
Index(['MatchId', 'Player1', 'Player2', 'Player3', 'Player4', 'Player5',
       'Player6', 'Player7', 'Player8', 'Player9', 'Player10', 'Win'],
      dtype='object')
['EUW1' 'NA1' 'KR']


In [4]:
print("Number of EUW1 matches: ",len(matches[matches["MatchId"].str.split("_").str[0] == "EUW1"]))
print("Number of NA1 matches: ",len(matches[matches["MatchId"].str.split("_").str[0] == "NA1"]))
print("Number of KR matches: ",len(matches[matches["MatchId"].str.split("_").str[0] == "KR"]))

Number of EUW1 matches:  3335
Number of NA1 matches:  3333
Number of KR matches:  3332


### Replaces empty positions as UNKNOWN

In [5]:
def replace_empty_position(player: list) -> list:
    if player[1] == "":
        player[1] = "UNKNOWN"
    return player

matches.update(matches.filter(like='Player').applymap(lambda x: replace_empty_position(eval(x))))
print(matches.loc[matches.MatchId == "EUW1_7266803946"])

C:\Users\jager\AppData\Local\Temp\ipykernel_12020\140216169.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  matches.update(matches.filter(like='Player').applymap(lambda x: replace_empty_position(eval(x))))


             MatchId                                            Player1  \
121  EUW1_7266803946  [champion_90, UNKNOWN, UNRANKED, UNRANKED, mas...   

                                               Player2  \
121  [champion_22, UNKNOWN, UNRANKED, UNRANKED, mas...   

                                               Player3  \
121  [champion_20, UNKNOWN, UNRANKED, UNRANKED, mas...   

                                               Player4  \
121  [champion_126, UNKNOWN, UNRANKED, UNRANKED, ma...   

                                               Player5  \
121  [champion_82, UNKNOWN, UNRANKED, UNRANKED, mas...   

                                               Player6  \
121  [champion_74, UNKNOWN, UNRANKED, UNRANKED, mas...   

                                               Player7  \
121  [champion_63, UNKNOWN, UNRANKED, UNRANKED, mas...   

                                               Player8  \
121  [champion_897, UNKNOWN, UNRANKED, UNRANKED, ma...   

                              

### Join Rank and Tier as one string, and caps the mastery level to 30

In [6]:
def merge_rank_tier_and_cap_mastery(player: list) -> list:
    #Save the champion and position in a variable
    champ_and_pos = player[:2]
    
    #Join the rank and tier as one str RANK_TIER
    rank = player[2]
    tier = player[3]
    if rank == 'UNRANKED' or tier == 'UNRANKED':
        rank_tier = "UNRANKED"
    else:
        rank_tier = rank.split("_")[1] + "_" + tier.split("_")[1]
    
    #Converts back mastery to int and caps it to 30
    mastery = player[4]
    mastery = int(mastery.split("_")[1])
    mastery = mastery if mastery <= 30 else 30
    
    return champ_and_pos + [rank_tier] + [mastery]

matches.update(matches.filter(like='Player').applymap(lambda x: merge_rank_tier_and_cap_mastery(x)))
print(matches.loc[matches.MatchId == "EUW1_7266803946"])

             MatchId                              Player1  \
121  EUW1_7266803946  [champion_90, UNKNOWN, UNRANKED, 0]   

                                 Player2                              Player3  \
121  [champion_22, UNKNOWN, UNRANKED, 0]  [champion_20, UNKNOWN, UNRANKED, 0]   

                                  Player4  \
121  [champion_126, UNKNOWN, UNRANKED, 0]   

                                 Player5                              Player6  \
121  [champion_82, UNKNOWN, UNRANKED, 0]  [champion_74, UNKNOWN, UNRANKED, 0]   

                                 Player7  \
121  [champion_63, UNKNOWN, UNRANKED, 0]   

                                  Player8  \
121  [champion_897, UNKNOWN, UNRANKED, 0]   

                                  Player9  \
121  [champion_110, UNKNOWN, UNRANKED, 0]   

                                Player10  Win  
121  [champion_81, UNKNOWN, UNRANKED, 0]    0  


C:\Users\jager\AppData\Local\Temp\ipykernel_12020\3365488880.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  matches.update(matches.filter(like='Player').applymap(lambda x: merge_rank_tier_and_cap_mastery(x)))


### Add Team ID to player arrays (0 or 1)

In [7]:
def add_team_id(players: list, column_name: str) -> list:
    #Add the team_id to the player list
    team_id = 0 if int(column_name.replace("Player","")) <= 5 else 1
    for i in range(len(players)):
        players[i] += [team_id]
    return players

matches.update(matches.filter(like='Player').apply(lambda x: add_team_id(players=x, column_name=x.name)))

### Unite player columns in one matrix

In [8]:
matches["Player"] = matches.filter(like='Player').apply(lambda x: list(x), axis=1)
matches.drop([f"Player{n}" for n in range(1,11)], axis=1, inplace=True)
print(matches.head())

           MatchId  Win                                             Player
0  EUW1_7266803975    0  [[champion_10, TOP, BRONZE_II, 4, 0], [champio...
1  EUW1_7266792870    0  [[champion_54, TOP, BRONZE_III, 2, 0], [champi...
2  EUW1_7265401358    1  [[champion_10, TOP, BRONZE_II, 4, 0], [champio...
3  EUW1_7263778355    1  [[champion_48, TOP, UNRANKED, 2, 0], [champion...
4  EUW1_7263730042    0  [[champion_82, TOP, BRONZE_III, 4, 0], [champi...


### Save df

In [9]:
matches.to_csv("data/matches_processed.csv", index=False)